# Яндекс Задания — наличие бабочек на фотографиях

In [ ]:
import json
import requests
from datetime import datetime, timedelta, timezone

TOKEN = ""
API_URL = "https://tasks.yandex.ru/api/v1"
HEADERS = {"Authorization": f"OAuth {TOKEN}", "Content-Type": "application/json"}

TRAIN_PASS = 80
EXAM_PASS = 80
HONEYPOT_PASS = 80
EXAM_OVERLAP = 20
WORK_OVERLAP = 3
WORK_REAL_PER_PAGE = 10
WORK_GOLDEN_PER_PAGE = 1
EXAM_PRICE = 1.2
WORK_PRICE = 0.8

WILL_EXPIRE = (datetime.now(timezone.utc) + timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
with open("data/train.json", encoding="utf-8") as file:
    train_data = json.load(file)

with open("data/exam.json", encoding="utf-8") as file:
    exam_data = json.load(file)

with open("data/dataset.json", encoding="utf-8") as file:
    photos_data = json.load(file)

with open("data/honeypots.json", encoding="utf-8") as file:
    honeypots_data = json.load(file)

EXAM_QUESTIONS = len(exam_data)
GOLDEN_HISTORY = EXAM_QUESTIONS

print("Тренировка:", len(train_data))
print("Экзамен, вопросов на странице:", EXAM_QUESTIONS)
print("Основные фотографии (ожидается 400):", len(photos_data))
print("Honeypots (ожидается 40):", len(honeypots_data))
print("Ожидаемых рабочих страниц:", len(photos_data) // WORK_REAL_PER_PAGE)

## 2. Полная инструкция для исполнителя

In [ ]:
instruction_html = """
<h2>Есть ли бабочка на фотографии?</h2>
<p>Вам нужно посмотреть на <b>фотографию</b> и определить, есть ли на фото бабочка</p>

<h3>Когда выбирать «Да, бабочка есть»</h3>
<ul>
<li>На фотографии есть хотя бы одна бабочка.</li>
<li>Если бабочка частично скрыта листьями, цветами, животными или другими предметами,
либо видна только её часть (например, крыло, тело и тд) — всё равно выбирайте «Да».</li>
<li><b>Моль и мотыльки</b> тоже считаются бабочками.</li>
<li>Размер бабочки, положение крыльев, тела и число бабочек не имеют значения.</li>
</ul>

<h3>Когда выбирать «Нет, бабочки нет»</h3>
<ul>
<li>На фото нет бабочки.</li>
<li>Есть только <b>другие насекомые</b>: например, жуки, пчёлы, мухи, стрекозы.</li>
<li>Есть только <b>гусеницы или куколки</b> бабочек, но нет взрослой бабочки.</li>
<li>Видна только <b>тень или силуэт</b> бабочки, но самой настоящей бабочки в кадре нет.</li>
</ul>



<h3>Примеры положительных ответов — «Да»</h3>
<p><b>[ПРИМЕР №1: бабочка, вид спереди]</b><br>
<img height="400" src="https://i.ibb.co/0pSV3QmM/front.png"></p>
<p><b>[ПРИМЕР №2: две бабочки]</b><br>
<img height="400" src="https://i.ibb.co/Rp1QqBQV/two-butterflies.jpg"></p>
<p><b>[ПРИМЕР ДА №3: моль / мотылёк]</b><br>
<img height="400" src="https://i.ibb.co/FqVBdfTB/moth.jpg"></p>
<p><b>[ПРИМЕР ДА №4: бабочка частично закрыта]</b><br>
<img height="400" src="https://i.ibb.co/CpB2hSXX/part.png"></p>

<h3>Примеры отрицательных ответов — «Нет»</h3>
<p><b>[ПРИМЕР НЕТ №1: гусеница / куколка]</b><br>
<img height="400" src="https://i.ibb.co/wNxf48nJ/doll.png"></p>

<h3>Горячие клавиши</h3>
<p><b>1 — Да, бабочка есть</b></p>
<p><b>2 — Нет, бабочки нет</b></p>
"""

## 3. Интерфейс и проект

In [ ]:
template = {
    "view": {
        "type": "view.list",
        "items": [
            {
                "type": "view.text",
                "content": "1 - Бабочка есть.\n2 - Бабочки нет\nЧастично закрытая бабочка, моль и мотылёк — ДА. Другие насекомые, гусеницы, куколки и тени — НЕТ."
            },
            {
                "type": "view.image",
                "label": "Рассмотрите фотографию",
                "url": {"type": "data.input", "path": "photo_url"},
                "maxWidth": 800,
                "popup": True
            },
            {
                "type": "field.radio-group",
                "label": "Есть ли на фото бабочка?",
                "data": {"type": "data.output", "path": "label"},
                "options": [
                    {"value": True, "label": "1 — Да, бабочка есть"},
                    {"value": False, "label": "2 — Нет, бабочки нет"}
                ],
                "validation": {"type": "condition.required", "hint": "Выберите один ответ"}
            }
        ]
    },
    "plugins": [
        {
            "type": "plugin.hotkeys",
            "1": {
                "type": "action.set",
                "data": {"type": "data.output", "path": "label"},
                "payload": True
            },
            "2": {
                "type": "action.set",
                "data": {"type": "data.output", "path": "label"},
                "payload": False
            }
        },
        {"type": "plugin.toloka", "layout": {"kind": "scroll", "taskWidth": 800}}
    ]
}

In [ ]:
project_json = {
    "public_name": "Бабочки на фотографиях",
    "public_description": "Определите, есть ли на фотографии бабочка.",
    "public_instructions": instruction_html,
    "task_spec": {
        "input_spec": {
            "photo_url": {"type": "url", "required": True, "hidden": False}
        },
        "output_spec": {
            "label": {"type": "boolean", "required": True, "hidden": False}
        },
        "view_spec": {
            "type": "tb",
            "config": json.dumps(template, ensure_ascii=False),
            "lock": {
                "core": "1.7.0",
                "view.list": "1.0.5",
                "view.text": "1.0.0",
                "view.image": "1.2.0",
                "field.radio-group": "1.1.10",
                "condition.required": "1.1.5",
                "plugin.toloka": "1.1.8",
                "plugin.hotkeys": "1.0.0",
                "action.set": "1.0.0"
            },
            "settings": {
                "showSkip": False,
                "showInstructions": True,
                "showTimer": True,
                "showSubmit": True,
                "showFullscreen": True
            }
        }
    },
    "assignments_issuing_type": "AUTOMATED",
    "assignments_automerge_enabled": False
}

project_response = requests.post(f"{API_URL}/projects", headers=HEADERS, json=project_json).json()
project_id = project_response["id"]
print("Проект:", project_id)
project_response

## 4. Создание тренировки


In [ ]:
training_json = {
    "project_id": project_id,
    "private_name": "Тренировка: бабочки",
    "may_contain_adult_content": False,
    "assignment_max_duration_seconds": 600,
    "mix_tasks_in_creation_order": False,
    "shuffle_tasks_in_task_suite": True,
    "training_tasks_in_task_suite_count": 1,
    "task_suites_required_to_pass": 1,
    "retry_training_after_days": 7,
    "inherited_instructions": True
}

training_response = requests.post(f"{API_URL}/trainings", headers=HEADERS, json=training_json).json()
training_id = training_response["id"]
print("Тренировка:", training_id)
training_response

In [ ]:
training_tasks_json = [
    {
        "pool_id": training_id,
        "input_values": {"photo_url": item["photo_url"]},
        "known_solutions": [{"output_values": {"label": item["label"]}, "correctness_weight": 1}],
        "message_on_unknown_solution": "Правильный ответ: " + ("На фотографии есть бабочка." if item["label"] else "На фотографии нет бабочки."),
        "infinite_overlap": True
    }
    for item in train_data
]

training_tasks_response = requests.post(
    f"{API_URL}/tasks", headers=HEADERS,
    params={"async_mode": "false"}, json=training_tasks_json
).json()
print("Загружено тренировочных заданий:", len(training_tasks_json))
training_tasks_response

## 5. Экзаменационный навык


In [ ]:
skill_json = {
    "name": f"Бабочки — экзамен — проект {project_id}",
    "private_comment": "Процент верных ответов на экзаменационные фотографии",
    "hidden": False
}

skill_response = requests.post(f"{API_URL}/skills", headers=HEADERS, json=skill_json).json()
exam_skill_id = skill_response["id"]
print("Экзаменационный навык:", exam_skill_id)
skill_response

## 6. Оплачиваемый экзамен — все вопросы на одной странице

In [ ]:
exam_pool_json = {
    "project_id": project_id,
    "private_name": "Экзамен",
    "public_description": "Оплачиваемый экзамен: определите, есть ли бабочка на фото.",
    "may_contain_adult_content": False,
    "will_expire": WILL_EXPIRE,
    "reward_per_assignment": EXAM_PRICE,
    "assignment_max_duration_seconds": 1800,
    "auto_accept_solutions": True,
    "defaults": {
        "default_overlap_for_new_task_suites": EXAM_OVERLAP,
        "default_overlap_for_new_tasks": EXAM_OVERLAP
    },
    "filter": {
        "and": [
            {"category": "profile", "key": "languages", "operator": "IN", "value": "RU"},
            {"category": "skill", "key": exam_skill_id, "operator": "EQ", "value": None}
        ]
    },
    "mixer_config": {
        "real_tasks_count": 0,
        "golden_tasks_count": EXAM_QUESTIONS,
        "training_tasks_count": 0,
        "shuffle_tasks_in_task_suite": True
    },
    "quality_control": {
        "training_requirement": {
            "training_pool_id": training_id,
            "training_passing_skill_value": TRAIN_PASS
        },
        "configs": [
            {
                "collector_config": {"type": "GOLDEN_SET"},
                "rules": [
                    {
                        "conditions": [
                            {"key": "golden_set_answers_count", "operator": "GTE", "value": EXAM_QUESTIONS}
                        ],
                        "action": {
                            "type": "SET_SKILL_FROM_OUTPUT_FIELD",
                            "parameters": {
                                "skill_id": exam_skill_id,
                                "from_field": "correct_answers_rate"
                            }
                        }
                    }
                ]
            }
        ]
    }
}

exam_pool_response = requests.post(f"{API_URL}/pools", headers=HEADERS, json=exam_pool_json).json()
exam_id = exam_pool_response["id"]
print("Экзаменационный пул:", exam_id)
exam_pool_response

In [ ]:
exam_tasks_json = [
    {
        "pool_id": exam_id,
        "input_values": {"photo_url": item["photo_url"]},
        "known_solutions": [{"output_values": {"label": item["label"]}, "correctness_weight": 1}],
        "overlap": EXAM_OVERLAP,
        "infinite_overlap": False
    }
    for item in exam_data
]

for start in range(0, len(exam_tasks_json), 500):
    exam_tasks_response = requests.post(
        f"{API_URL}/tasks", headers=HEADERS,
        params={"async_mode": "false"}, json=exam_tasks_json[start:start + 500]
    ).json()
    print("Загружено экзаменационных заданий:", min(start + 500, len(exam_tasks_json)))

## 7. Основной пул: 10 обычных + 1 honeypot


In [ ]:
work_pool_json = {
    "project_id": project_id,
    "private_name": "Основная разметка: бабочки",
    "public_description": "Определите, есть ли на фотографии бабочка.",
    "may_contain_adult_content": False,
    "will_expire": WILL_EXPIRE,
    "reward_per_assignment": WORK_PRICE,
    "assignment_max_duration_seconds": 1800,
    "auto_accept_solutions": True,
    "defaults": {
        "default_overlap_for_new_task_suites": WORK_OVERLAP,
        "default_overlap_for_new_tasks": WORK_OVERLAP
    },
    "filter": {
        "and": [
            {"category": "profile", "key": "languages", "operator": "IN", "value": "RU"},
            {"category": "skill", "key": exam_skill_id, "operator": "GTE", "value": EXAM_PASS}
        ]
    },
    "mixer_config": {
        "real_tasks_count": WORK_REAL_PER_PAGE,
        "golden_tasks_count": WORK_GOLDEN_PER_PAGE,
        "training_tasks_count": 0,
        "shuffle_tasks_in_task_suite": True
    },
    "quality_control": {
        "configs": [
            {
                "collector_config": {
                    "type": "GOLDEN_SET",
                    "parameters": {"history_size": GOLDEN_HISTORY}
                },
                "rules": [
                    {
                        "conditions": [
                            {"key": "golden_set_answers_count", "operator": "GTE", "value": EXAM_QUESTIONS},
                            {"key": "golden_set_correct_answers_rate", "operator": "LT", "value": HONEYPOT_PASS}
                        ],
                        "action": {
                            "type": "RESTRICTION_V2",
                            "parameters": {
                                "scope": "POOL",
                                "duration_unit": "DAYS",
                                "duration": 10,
                                "private_comment": "Доля правильных контрольных ответов ниже 80%"
                            }
                        }
                    }
                ]
            }
        ]
    }
}

work_pool_response = requests.post(f"{API_URL}/pools", headers=HEADERS, json=work_pool_json).json()
work_id = work_pool_response["id"]
print("Боевой пул:", work_id)
work_pool_response

### 7.1. JSON боевого пула

In [ ]:
work_tasks_json = [
    {
        "pool_id": work_id,
        "input_values": {"photo_url": item["photo_url"]},
        "overlap": WORK_OVERLAP
    }
    for item in photos_data
]

print("Обычных заданий для загрузки:", len(work_tasks_json))

In [ ]:
for start in range(0, len(work_tasks_json), 500):
    work_tasks_response = requests.post(
        f"{API_URL}/tasks", headers=HEADERS,
        params={"async_mode": "false"}, json=work_tasks_json[start:start + 500]
    ).json()
    print("Загружено обычных заданий:", min(start + 500, len(work_tasks_json)))

### 7.2. JSON контрольных фотографий

In [ ]:
honeypot_tasks_json = [
    {
        "pool_id": work_id,
        "input_values": {"photo_url": item["photo_url"]},
        "known_solutions": [{"output_values": {"label": item["label"]}, "correctness_weight": 1}],
        "infinite_overlap": True
    }
    for item in honeypots_data
]

print("Контрольных заданий для загрузки:", len(honeypot_tasks_json))

In [ ]:
for start in range(0, len(honeypot_tasks_json), 500):
    honeypot_tasks_response = requests.post(
        f"{API_URL}/tasks", headers=HEADERS,
        params={"async_mode": "false"}, json=honeypot_tasks_json[start:start + 500]
    ).json()
    print("Загружено контрольных заданий:", min(start + 500, len(honeypot_tasks_json)))

## 8. Проверка перед запуском и модерация

In [ ]:
exam_validate = requests.get(
    f"https://tasks.yandex.ru/api/new/requester/pools/{exam_id}/validate",
    headers=HEADERS
).json()
exam_validate

In [ ]:
work_validate = requests.get(
    f"https://tasks.yandex.ru/api/new/requester/pools/{work_id}/validate",
    headers=HEADERS
).json()
work_validate

In [ ]:
exam_moderation_json = {"status": "READY"}
exam_moderation_response = requests.put(
    f"https://tasks.yandex.ru/api/new/requester/poolModeration/{exam_id}",
    headers=HEADERS, json=exam_moderation_json
).text
print(exam_moderation_response)

In [ ]:
work_moderation_json = {"status": "READY"}
work_moderation_response = requests.put(
    f"https://tasks.yandex.ru/api/new/requester/poolModeration/{work_id}",
    headers=HEADERS, json=work_moderation_json
).text
print(work_moderation_response)

## 9. Открытие 

In [ ]:
training_open_response = requests.post(
    f"{API_URL}/trainings/{training_id}/open", headers=HEADERS
).text
training_open_response

In [ ]:
exam_open_response = requests.post(
    f"{API_URL}/pools/{exam_id}/open", headers=HEADERS
).text
exam_open_response

In [ ]:
work_open_response = requests.post(
    f"{API_URL}/pools/{work_id}/open", headers=HEADERS
).text
work_open_response